# AI Agent Skills: Build a Python Developer Agent

In this tutorial, we'll build a Python developer agent and progressively add two Skills:

1. **`explain-code`** — a beginner-friendly, step-by-step code explanation workflow.
2. **`code-review`** — a systematic review workflow with severity levels, a reference checklist, a JSON asset, and a helper script.

We'll compare responses before and after Skills using the same requests.

> OpenRouter provides model access. Skill discovery, selection, and loading are implemented by our lightweight Python runtime in this notebook.

## 1. Setup

Install dependencies if needed:

```bash
pip install requests python-dotenv markdown pycodestyle
```

Create a `.env` file beside the notebook:
* OPEN_ROUTER_KEY=your_openrouter_api_key

In [1]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path
import tempfile

import markdown
from IPython.display import Markdown, HTML

import requests
from dotenv import load_dotenv, find_dotenv

is_env_var_loaded = load_dotenv(find_dotenv())
print(f"Is environment variables loaded?: {'Yes' if is_env_var_loaded else 'No'}")

MODEL = "deepseek/deepseek-v4.1-flash" ## Feel free to try other LLMs from OpenRouter, the same code will work just fine.
OPEN_ROUTER_KEY = os.environ.get("OPEN_ROUTER_KEY")
SKILLS_DIR = Path("skills")
DEFAULT_SYSTEM_PROMPT = "You are a helpful Python developer assistant."

headers = {
    "Authorization": f"Bearer {OPEN_ROUTER_KEY}",
    "Content-Type": "application/json",
}
CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"

Is environment variables loaded?: Yes


#### LLM Call

In [2]:
def call_model(messages, temperature=0.2):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
    }
    response = requests.post(
        CHAT_URL,
        headers=headers,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

## 2. Helper Functions

Helper functions for agent call, discover skills, load skill resources, select skill & execute scripts.

### 2.1 Discover All Available Skills

In [24]:
def read_skill_metadata(skill_dir):
    skill_file = skill_dir / "SKILL.md"
    content = skill_file.read_text(encoding="utf-8")
    if not content.startswith("---"):
        raise ValueError(f"Missing frontmatter in {skill_file}")
    parts = content.split("---", 2)
    if len(parts) < 3:
        raise ValueError(f"Invalid frontmatter in {skill_file}")

    metadata = {}
    for line in parts[1].splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            metadata[key.strip()] = value.strip()

    return {
        "name": metadata.get("name", skill_dir.name),
        "description": metadata.get("description", ""),
        "path": str(skill_file),
    }


def discover_skills():
    if not os.path.exists(SKILLS_DIR):
        return []
    found = []
    for child in sorted(SKILLS_DIR.iterdir()):
        if child.is_dir() and (child / "SKILL.md").exists():
            found.append(read_skill_metadata(child))
    return found

In [25]:
discover_skills()

[{'name': 'code-review',
  'description': 'Review Python code for correctness, edge cases, error handling, readability, performance, and tests. Use when a user asks for a code review or improvement recommendations.',
  'path': 'skills/code-review/SKILL.md'},
 {'name': 'explain-code',
  'description': 'Explain Python code step by step for developers who want to understand what code does, how it works, and what output it produces.',
  'path': 'skills/explain-code/SKILL.md'}]

### 2.2 Load Skill Resources 
* Loads **Skill.md**
* Files inside of **references** Folder
* Files inside of **assets** folder.

Combines all three and creates one context.

In [4]:
MAX_FILE_CHARS = 15_000
MAX_SKILL_CONTEXT_CHARS = 40_000

def load_skill_resources(skill):
    """Load SKILL.md and supporting reference and asset files."""
    skill_file = Path(skill["path"])
    skill_dir = skill_file.parent
    sections = []

    if skill_file.is_file():
        sections.append(
            "## SKILL.md\n\n"
            + skill_file.read_text(encoding="utf-8")
        )

    # Scripts are executed separately by execute_skill_scripts().
    for directory_name in ["references", "assets"]:
        resource_dir = skill_dir / directory_name

        if not resource_dir.is_dir():
            continue

        for file_path in sorted(resource_dir.rglob("*")):
            if not file_path.is_file():
                continue

            if file_path.suffix.lower() not in {
                ".md", ".txt", ".json", ".yaml", ".yml",
                ".toml", ".csv", ".sql"
            }:
                continue

            try:
                content = file_path.read_text(encoding="utf-8")
            except (OSError, UnicodeDecodeError):
                continue

            relative_path = file_path.relative_to(skill_dir)

            sections.append(
                f"## Resource: {relative_path}\n\n{content}"
            )

    return "\n\n".join(sections)

### 2.3 Select Skill

In [5]:
def select_skill(user_request, skills):
    if not skills:
        return None, "No Skills available."
    options = [
        {"name": item["name"], "description": item["description"]}
        for item in skills
    ]
    router_prompt = (
        "Choose the single most relevant Skill for the user's request. "
        "If none is relevant, choose null. Return only JSON: "
        '{"skill": "skill-name-or-null", "reason": "brief reason"}.'
    )
    result = call_model([
        {"role": "system", "content": router_prompt},
        {"role": "user", "content": json.dumps({
            "request": user_request,
            "available_skills": options,
        }, indent=2)},
    ])

    cleaned = result.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        decision = json.loads(cleaned)
    except json.JSONDecodeError:
        return None, f"Could not parse Skill selection: {result}"

    name = decision.get("skill")
    valid_names = {item["name"] for item in skills}
    if name not in valid_names:
        return None, decision.get("reason", "No relevant Skill selected.")
    return next(item for item in skills if item["name"] == name), decision.get("reason", "")

### 2.4 Execute Skill Scripts Helper

In [6]:
def extract_code_from_request(user_request):
    """Extract Python code from a Markdown code block in the user request."""
    pattern = r"```(?:python|py)?\s*\n(.*?)```"
    matches = re.findall(pattern, user_request, flags=re.IGNORECASE | re.DOTALL)

    if not matches:
        return None

    # Use the first code block as the code to inspect.
    return matches[0].strip()

In [7]:
def execute_skill_scripts(skill, code):
    """Run all Python scripts in a Skill's scripts directory."""
    scripts_dir = Path(skill["path"]).parent / "scripts"

    if not scripts_dir.is_dir() or not code:
        return ""

    scripts = sorted(scripts_dir.glob("*.py"))
    if not scripts:
        return ""

    results = []

    # Make the user's code available to every script.
    with tempfile.TemporaryDirectory() as temp_dir:
        code_path = Path(temp_dir) / "user_code.py"
        code_path.write_text(code, encoding="utf-8")

        for script_path in scripts:
            try:  
                result = subprocess.run(
                    [sys.executable, str(script_path), str(code_path)], ## sys.executable -> python binary path
                    capture_output=True,
                    text=True,
                    timeout=30,
                    check=False,
                )

                results.append(
                    f"### {script_path.name}\n"
                    f"Exit code: {result.returncode}\n"
                    f"Output:\n{result.stdout or '(No output)'}\n"
                    f"Errors:\n{result.stderr or '(No errors)'}"
                )
                if result.returncode == 0:
                    print(f"Executed {script_path} successfully...")
            except subprocess.TimeoutExpired:
                results.append(
                    f"### {script_path.name}\n"
                    "Execution timed out after 30 seconds."
                )
            except OSError:
                results.append(
                    f"### {script_path.name}\n"
                    "OS Error."
                )

    return "\n\n".join(results)

## 3. Baseline: ask the agent without Skills

We'll use the same request before and after adding a Skill. The model may still give a good answer without Skills; our comparison is about whether the response follows a more deliberate, reusable workflow.

In [13]:
explain_code = '''
def calculate_average(numbers):
    if not numbers:
        return 0

    total = sum(numbers)
    count = len(numbers)

    return total / count
'''

explain_request = f"""Explain this Python code to me:

```python
{explain_code}
```"""

In [12]:
def ask_agent(user_request):
    """Answer a user request with an optional dynamically loaded Skill."""

    # 1. Discover the available Skills.
    skills = discover_skills()

    # 2. Select a relevant Skill, if any.
    selected_skill, reason = select_skill(user_request, skills)
    print("Skill : ", selected_skill)
    print("Reason: ", reason)
    
    # 3. Always start with the same default system prompt.
    messages = [
        {
            "role": "system",
            "content": DEFAULT_SYSTEM_PROMPT
        }
    ]

    # 4. Load Skill instructions and resources only when selected.
    if selected_skill is not None:
        skill_context = load_skill_resources(selected_skill)

        # 5. Extract user code and execute Skill scripts when available.
        user_code = extract_code_from_request(user_request)
        script_results = execute_skill_scripts(
            selected_skill,
            user_code
        )

        if script_results:
            skill_context += (
                "\n\n## Results from Skill Scripts\n\n"
                "Use the following results when answering the user. "
                "Distinguish successful checks from errors.\n\n"
                f"{script_results}"
            )

        messages.append({
            "role": "system",
            "content": (
                "Follow the instructions in the selected Skill and use its "
                "supporting resources and script results when relevant.\n\n"
                f"Selected Skill: {selected_skill['name']}\n"
                f"Selection reason: {reason}\n\n"
                f"{skill_context}"
            )
        })

    # 6. Send the user's request through the common model method.
    messages.append({
        "role": "user",
        "content": user_request
    })

    answer = call_model(messages)

    return answer

In [14]:
baseline_explanation = ask_agent(explain_request)

display(Markdown("---"))
display(Markdown(baseline_explanation))

Skill :  None
Reason:  No Skills available.


---

This Python function calculates the average (mean) of a list of numbers.

```python
def calculate_average(numbers):
    if not numbers:
        return 0

    total = sum(numbers)
    count = len(numbers)

    return total / count
```

Step by step:

1. **Define the function**
   ```python
   def calculate_average(numbers):
   ```
   The function is named `calculate_average` and takes one argument, `numbers`, which is expected to be a list of numeric values.

2. **Handle an empty list**
   ```python
   if not numbers:
       return 0
   ```
   In Python, an empty list is considered “falsy.” So if `numbers` is empty, the function returns `0` immediately. This avoids dividing by zero later.

3. **Calculate the total**
   ```python
   total = sum(numbers)
   ```
   `sum(numbers)` adds up all the numbers in the list.

4. **Count how many numbers there are**
   ```python
   count = len(numbers)
   ```
   `len(numbers)` returns the number of items in the list.

5. **Return the average**
   ```python
   return total / count
   ```
   The average is the total divided by the count.

Example:

```python
calculate_average([10, 20, 30])
```

- `total = 10 + 20 + 30 = 60`
- `count = 3`
- `return 60 / 3 = 20.0`

So the result is:

```python
20.0
```

And for an empty list:

```python
calculate_average([])
```

It returns:

```python
0
```

In short: this function safely computes the arithmetic mean of a list of numbers, returning `0` if the list is empty.

-------------------------------------------

## 4. Skill: explain-code

The **Skill.md** file is present in `skills/explain-code` folder.

## 5. Run the agent with the selected Skill

Once a Skill is selected, the runtime loads its `SKILL.md` instructions.

In [15]:
explanation_with_skill = ask_agent(explain_request)

display(Markdown("---"))
display(Markdown(explanation_with_skill))

Skill :  {'name': 'explain-code', 'description': 'Explain Python code step by step for developers who want to understand what code does, how it works, and what output it produces.', 'path': 'skills/explain-code/SKILL.md'}
Reason:  The user asks for a step-by-step explanation of Python code, which directly matches the explain-code skill.


---

### 1. Overview
- This code defines a function `calculate_average` that computes the arithmetic mean (the sum of the values divided by how many values there are) of a collection of numbers.
- It is designed to be safe for empty input: instead of crashing with a division-by-zero error, it returns `0`.

### 2. How It Works
1. `def calculate_average(numbers):` defines a function named `calculate_average` that takes one parameter, `numbers`, which is expected to be a collection of numeric values (such as a list or tuple).
2. `if not numbers:` uses Python's truthiness rules — an empty list, empty tuple, `None`, or `0` all evaluate to `False`, so `not numbers` becomes `True` for those values. This is the guard clause that handles the "nothing to average" case.
3. `return 0` exits the function immediately when the guard is triggered, preventing the later division from ever being reached with a zero count.
4. `total = sum(numbers)` calls the built-in `sum()` function, which iterates over `numbers` and adds every element together, storing the result in `total`. This works because `sum()` starts from `0` and applies `+` to each element in turn.
5. `count = len(numbers)` calls the built-in `len()` function, which asks the collection how many elements it contains. For lists and tuples this is a stored attribute lookup, so it is effectively instant.
6. `return total / count` performs true division (`/`), which in Python 3 always produces a `float`, even when both operands are integers. This value is the average and is handed back to the caller.

### 3. Example Walkthrough
- Input: `calculate_average([4, 8, 6])`
- Step 1 — `if not numbers:` → the list has three elements, so it is truthy, and `not numbers` is `False`. The guard is skipped.
- Step 2 — `total = sum([4, 8, 6])` → `0 + 4 = 4`, `4 + 8 = 12`, `12 + 6 = 18`, so `total` becomes `18`.
- Step 3 — `count = len([4, 8, 6])` → `count` becomes `3`.
- Step 4 — `return 18 / 3` → returns `6.0`.
- Contrasting input: `calculate_average([])`
- Step 1 — `not []` is `True`, so the function returns `0` immediately; `sum()` and `len()` are never called.

### 4. Expected Output
- The function itself prints nothing; it only returns a value. The output below is what you would see if you called it and printed the results (derived by tracing the code, not by executing it):

```python
print(calculate_average([4, 8, 6]))   # 6.0
print(calculate_average([10, 20]))    # 15.0
print(calculate_average([5]))         # 5.0
print(calculate_average([]))          # 0
```

```text
6.0
15.0
5.0
0
```

- Note that the first three results are floats (`6.0`, not `6`) because `/` always returns a float in Python 3, while the empty-input case returns the integer `0`.

### 5. Time and Space Complexity
- Time complexity: **O(n)**, where *n* is the number of elements in `numbers`. The `sum()` call must visit every element once, so the running time grows linearly with the input size. `len()` and the division are O(1) constant-time operations, and the guard check is also O(1) for standard containers.
- Space complexity: **O(1)** extra space. Only two scalar variables (`total` and `count`) are created regardless of input size; `sum()` accumulates into a single running value rather than building a new collection.

### 6. Important Notes
- **Inconsistent return type:** the empty case returns the integer `0`, while all other cases return a `float`. Code that consumes this function should not assume a single numeric type.
- **Truthiness, not a length check:** `if not numbers` also triggers for `None`, `0`, an empty string, or an empty tuple — not just an empty list. Passing `None` silently returns `0` instead of raising an error, which can hide bugs.
- **Requires `len()` support:** because `len(numbers)` is called, the argument must be a sized iterable (list, tuple, `range`, etc.). A generator or iterator would raise `TypeError` even though `sum()` alone could handle it.
- **Non-numeric elements:** if the collection contains strings or other non-addable types, `sum()` raises `TypeError`; the function performs no validation or type coercion.
- **No rounding:** the raw floating-point result is returned, so values like `calculate_average([1, 2])` yield `1.5` and repeating decimals may show floating-point representation artifacts.
- **Assumption:** the code assumes "average" means the arithmetic mean of all elements, with equal weight given to each value.

## 6. Compare the explanation before and after the Skill

In [16]:
without_skill_html = markdown.markdown(
    baseline_explanation,
    extensions=["fenced_code", "tables"]
)

with_skill_html = markdown.markdown(
    explanation_with_skill,
    extensions=["fenced_code", "tables"]
)

display(HTML(f"""
<div style="display: flex; gap: 16px; align-items: flex-start;">

    <div style="flex: 1; min-width: 0; padding: 16px;
                border: 1px solid #ccc; border-radius: 10px;">
        <h3>Without Skill</h3>
        {without_skill_html}
    </div>

    <div style="flex: 1; min-width: 0; padding: 16px;
                border: 1px solid #ccc; border-radius: 10px;">
        <h3>With Skill</h3>
        {with_skill_html}
    </div>

</div>
"""))

## 7. Skill: Code Review

This function handles empty lists explicitly, but its behavior for invalid inputs and its choice to return 0 for an empty list are worth reviewing.

In [19]:
explain_code = '''
def calculate_average(numbers):
    if not numbers:
        return 0

    total = sum(numbers)
    count = len(numbers)

    return total / count
'''

review_request = f"""Please review this Python function and suggest improvements.

```python
{explain_code}
```"""

### 7.1 Without Skill

In [20]:
baseline_review = ask_agent(review_request)

display(Markdown("---"))
display(Markdown(baseline_review))

Skill :  None
Reason:  The available skill explains code, but the request asks for a review and improvement suggestions, which is not covered.


---

The function is correct for a non-empty list/tuple of numbers, but there are a few improvements worth making.

### Main suggestions

1. **Clarify the empty-input behavior**
   - Returning `0` for an empty list can hide bugs or missing data.
   - Prefer raising an exception, or return `None` if the caller must handle “no data” explicitly.
   - If `0` is intentional, document it clearly.

2. **Add type hints and a docstring**
   - Makes the function easier to use and maintain.

3. **Use `math.fsum` for floating-point accuracy**
   - `sum()` can accumulate floating-point rounding errors.
   - `math.fsum()` is more accurate for floats.

4. **Decide whether to accept only sequences or any iterable**
   - `if not numbers` and `len(numbers)` require a sequence like `list` or `tuple`.
   - If you want to accept generators, convert to a list first.

5. **Consider `statistics.mean` / `statistics.fmean`**
   - `statistics.fmean` is accurate for floats.
   - `statistics.mean` preserves `Decimal` / `Fraction` types better.

### Improved version: strict empty handling

```python
from collections.abc import Sequence
from math import fsum


def calculate_average(numbers: Sequence[float]) -> float:
    """Return the arithmetic mean of a non-empty sequence of numbers.

    Args:
        numbers: A non-empty sequence of numeric values.

    Raises:
        ValueError: If numbers is empty.
    """
    if not numbers:
        raise ValueError("numbers must not be empty")

    return fsum(numbers) / len(numbers)
```

### If you must keep returning `0` for empty input

```python
from collections.abc import Sequence
from math import fsum


def calculate_average(numbers: Sequence[float]) -> float:
    """Return the arithmetic mean, or 0.0 if numbers is empty."""
    if not numbers:
        return 0.0

    return fsum(numbers) / len(numbers)
```

### If you want to accept any iterable

```python
from collections.abc import Iterable
from math import fsum


def calculate_average(numbers: Iterable[float]) -> float:
    """Return the arithmetic mean of a non-empty iterable of numbers."""
    values = list(numbers)

    if not values:
        raise ValueError("numbers must not be empty")

    return fsum(values) / len(values)
```

### Using the standard library

If you prefer not to implement the averaging logic yourself:

```python
from statistics import fmean, StatisticsError


def calculate_average(numbers):
    try:
        return fmean(numbers)
    except StatisticsError:
        return 0.0
```

For most cases, the strict version with `math.fsum` and a clear `ValueError` is the safest improvement.

### 7.2 With Skill

In [22]:
review_with_skill = ask_agent(review_request)

display(Markdown("---"))
display(Markdown(review_with_skill))

Skill :  {'name': 'code-review', 'description': 'Review Python code for correctness, edge cases, error handling, readability, performance, and tests. Use when a user asks for a code review or improvement recommendations.', 'path': 'skills/code-review/SKILL.md'}
Reason:  The user asks to review a Python function and suggest improvements, which matches the code-review skill.
Executed skills/code-review/scripts/check_style.py successfully...
Executed skills/code-review/scripts/inspect_python.py successfully...


---

## 1. Summary

- The function is small, readable, and correctly computes the arithmetic mean for a non-empty sequence of numbers; the empty-input guard also prevents a `ZeroDivisionError`.
- The main risks are the ambiguous `return 0` for empty input (0 is a valid average, so it can mask errors) and the lack of input validation, which lets `None` or non-numeric values fail silently or with confusing exceptions.

## 2. Findings

- **Severity:** Medium
  - **Category:** Correctness / Edge Case
  - **Issue:** Empty input returns `0`, which is indistinguishable from a legitimate average of `0` (e.g., `[0]` or `[-1, 1]`). This is a confirmed behavior, though whether it is a "bug" depends on intent.
  - **Impact:** Callers cannot tell "no data" from "average is zero," which can silently corrupt downstream calculations or reporting.
  - **Location:** `if not numbers: return 0` (lines 2–3).

- **Severity:** Medium
  - **Category:** Edge Case / Error Handling
  - **Issue:** `if not numbers` treats any falsy value as empty. Passing `None` returns `0` silently instead of raising, hiding a likely caller mistake.
  - **Impact:** A `None` argument (common when data is missing) produces a plausible-looking result rather than a clear error, making bugs hard to trace.
  - **Location:** `if not numbers:` (line 2).

- **Severity:** Low
  - **Category:** Edge Case / Error Handling
  - **Issue:** No validation of element types. A non-iterable argument (e.g., an `int`) or a sequence containing non-numeric values raises `TypeError` from `sum()`.
  - **Impact:** Errors surface from inside `sum()` with messages that may not clearly point to the caller's input. This is a potential risk, not a confirmed defect, since letting the caller handle it is a valid design.
  - **Location:** `total = sum(numbers)` (line 5).

- **Severity:** Low
  - **Category:** Style
  - **Issue:** `W292 no newline at end of file` reported by the style checker.
  - **Impact:** Minor; some tooling and diffs prefer a trailing newline. No functional effect.
  - **Location:** Line 8 (end of file).

- **Severity:** Low
  - **Category:** Readability
  - **Issue:** The empty-input policy is undocumented; the docstring is absent.
  - **Impact:** Callers must read the code to learn the empty-input contract.
  - **Location:** Function definition (line 1).

- **Note:** No correctness, performance, or complexity issues were found in the core computation. `sum`/`len` is O(n) and appropriate; the division-by-zero path is correctly guarded.

## 3. Recommended Fix

- Make the empty-input contract explicit and validate the argument type. A small, targeted change:

```python
def calculate_average(numbers):
    """Return the arithmetic mean of a non-empty sequence of numbers.

    Raises:
        ValueError: If `numbers` is empty.
        TypeError: If `numbers` is not an iterable of numbers.
    """
    if numbers is None:
        raise TypeError("numbers must be an iterable of numbers, not None")

    total = sum(numbers)
    count = len(numbers)

    if count == 0:
        raise ValueError("calculate_average() requires at least one number")

    return total / count
```

- Key changes:
  - Replaced the falsy check with an explicit `None` check so `None` raises a clear `TypeError` instead of returning `0`.
  - Moved the empty check after `len()` and raise `ValueError`, so "no data" is distinguishable from an average of `0`.
  - Added a docstring documenting the contract.
  - Added a trailing newline (fixes `W292`).
- If returning `0` for empty input is intentional for your use case, keep it but document it in the docstring and consider returning `None` instead to signal "no data." The choice depends on caller expectations.

## 4. Tests to Add

- Normal case: `calculate_average([1, 2, 3])` returns `2.0`.
- Single element: `calculate_average([5])` returns `5.0`.
- Negative and mixed values: `calculate_average([-1, 1])` returns `0.0` (verifies a real zero is not confused with empty input).
- Floats: `calculate_average([1.5, 2.5])` returns `2.0`.
- Empty input: `calculate_average([])` raises `ValueError` (or returns the documented sentinel, per your chosen contract).
- `None` input: `calculate_average(None)` raises `TypeError`.
- Non-numeric elements: `calculate_average([1, "a"])` raises `TypeError`.
- Non-iterable input: `calculate_average(5)` raises `TypeError`.
- Generator/iterator input: confirm behavior, since `len()` requires a sized object (a generator would raise `TypeError` here) — decide whether to support it.

## 5. Limitations

- No tests were executed; the test cases above are recommendations, not verified results.
- The style check (`check_style.py`) was run and reported one violation (`W292`, line 8); this is an automated static check, not a test run.
- The static inspection (`inspect_python.py`) confirmed 8 lines, one function, no `try/except`, and no syntax errors; it does not evaluate runtime behavior.
- The intended behavior for empty input is assumed to be "no valid average"; if returning `0` is deliberate, the severity of the first finding drops to Low.
- Surrounding code and callers were not provided, so the impact of the empty-input and `None` behaviors on the wider system could not be assessed.

## 8. Compare code review before and after the Skill

In [23]:
without_skill_html = markdown.markdown(
    baseline_review,
    extensions=["fenced_code", "tables"]
)

with_skill_html = markdown.markdown(
    review_with_skill,
    extensions=["fenced_code", "tables"]
)

display(HTML(f"""
<div style="display: flex; gap: 16px; align-items: flex-start;">

    <div style="flex: 1; min-width: 0; padding: 16px;
                border: 1px solid #ccc; border-radius: 10px;">
        <h3>Without Skill</h3>
        {without_skill_html}
    </div>

    <div style="flex: 1; min-width: 0; padding: 16px;
                border: 1px solid #ccc; border-radius: 10px;">
        <h3>With Skill</h3>
        {with_skill_html}
    </div>

</div>
"""))

## 9. Key takeaways

- **The model** provides general language and coding capabilities.
- **A Skill** provides reusable instructions and optional supporting resources for a specific workflow.
- **The runtime** discovers Skills, selects one, loads its instructions, and decides which resources to provide.
- **OpenRouter** provides model access; it does not automatically implement this custom Skill runtime.
- Skills can encourage more consistent structure, but they do not guarantee a better answer every time.

### Production considerations
This is an educational implementation. For production, use robust frontmatter parsing, validate model output, add tests and logging, define Skill permissions, and avoid executing scripts from untrusted sources. Treat Skill files as trusted project code.